In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

# Ensure Experiment Workspace Directories

Creates the parent directory tree required by MLflow experiment resources.
MLflow experiment creation requires all parent directories to pre-exist.

Target path: `/Workspace/.experiments/{catalog}/{schema}/`

Idempotent - safe to re-run. Must run BEFORE `bundle deploy` creates
the experiment resource on first deployment to a new target.

In [0]:
dbutils.widgets.text("catalog", "", "Catalog Name")
dbutils.widgets.text("schema", "", "Schema Name")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

assert catalog, "catalog parameter is required"
assert schema, "schema parameter is required"

print(f"Catalog: {catalog}")
print(f"Schema:  {schema}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Build the directory path segments
experiment_root = "/Workspace/.experiments"
dirs_to_create = [
    experiment_root,
    f"{experiment_root}/{catalog}",
    f"{experiment_root}/{catalog}/{schema}",
]

for directory in dirs_to_create:
    try:
        w.workspace.mkdirs(directory)
        print(f"  Ensured: {directory}")
    except Exception as e:
        if "ALREADY_EXISTS" in str(e) or "already exists" in str(e).lower():
            print(f"  Exists:  {directory}")
        else:
            raise RuntimeError(f"Failed to create {directory}: {e}") from e

print(f"\nDone. Experiment directory ready: {dirs_to_create[-1]}/")